In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_conn_Eensemble_withLessCfg')

projs=['P0', 'Px', 'Py', 'Pz']
inserts=['tt', 'tx', 'ty', 'tz', 'xx', 'xy', 'xz', 'yy', 'yz', 'zz']
enss=['e']

app='_cfgs_conn_Giannis_fine'

# 2pt

In [2]:
ens2c2pt={}; ens2moms_2pt={}; ens2c2pt0={}; ens2Njk={}
for ens in enss:
    basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
    path=f'{basepath}disc_2pt{app}.h5'
    with h5py.File(path) as f:
        moms_2pt=yu.moms2list(f['moms'])
        c2pt=yu.jackknife(np.real(f['data/N_N'][:,:,:]))
        
    ens2moms_2pt[ens]=moms_2pt
    ens2c2pt[ens]=c2pt
    ens2c2pt0[ens]=c2pt[:,:,moms_2pt.index([0,0,0])]
    ens2Njk[ens]=len(c2pt)

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [3]:
# global parameters
ens2tminss={
        'b':[range(8,25+1),range(1,10+1),range(1,4+1)],
        'c':[range(8,29+1),range(1,16+1),range(1,5+1)],
        'd':[range(8,33+1),range(1,18+1),range(1,6+1)],
        # 'e':[range(8,39+1),range(1,18+1),range(1,5+1)],
        'e':[range(8,39+1),range(1,16+1),range(1,5+1)],
    }
ens2selections={
    'b':{'1st':20,'2st':7,'3st':3},
    'c':{'1st':21,'2st':8,'3st':3},
    'd':{'1st':24,'2st':9,'3st':3},
    'e':{'1st':32,'2st':11,'3st':4},
}

ens2tminss_large={
        'b':[range(8,25+1),range(1,9+1),range(1,1+1)],
        'c':[range(8,29+1),range(1,10+1),range(1,1+1)],
        'd':[range(8,33+1),range(1,11+1),range(1,1+1)],
        'e':[range(8,39+1),range(1,13+1),range(1,1+1)],
    } # used for large momenta

In [4]:
overwrite=False
ens2DmN={'b':(-4.44,0.27),'c':(-1.44,0.24),'d':(-4.95,0.28),'e':(-0.845,0.022)}
ens2DmN={ens:yu.jackknife_pseudo(ens2DmN[ens][0],ens2DmN[ens][1],ens2Njk[ens])[:,0] for ens in enss}

mN_exp=(yu.m_proton+yu.m_neutron)/2

figs=[]; ens2pars_jk_meff1st={}; ens2pars_jk_meff2st={}; ens2pars_jk_meff3st={}
for ens in enss:
    meff=yu.jackmap(yu.c2pt2meff,ens2c2pt0[ens])
    tminss=ens2tminss[ens]

    # tmins=[1.6,0.6,0.2]
    # tmins=[1.6,0.6,0.2]
    # tmins=[t*yu.ens2a['b'] for t in [20,7,3]]
    # selections={f'{i+1}st':yu.find_t_cloest(tmins[i],yu.ens2a[ens]) for i in range(3)}
    selections=ens2selections[ens]
    # selections={}
    print(ens,selections)
    
    fitss_2pt=yu.doFits_meff_nst(meff,tminss,[0.4,0.5,2,0.8,1],downSampling=1,label=f'meff_{ens}',overwrite=overwrite)
    fig,axd,result=yu.makePlot_2pt_SimoneStyle(meff,fitss_2pt,xunit=yu.ens2a[ens],yunit=yu.ens2aInv[ens]/1000,E0_ref=mN_exp/1000,ylims='std_N',\
        selection=selections)
    fig.suptitle(yu.ens2full[ens])
    yu.finalizePlot(closeQ=True)
    figs.append(fig) 
    
    ens2pars_jk_meff1st[ens]=result['1st']
    ens2pars_jk_meff2st[ens]=result['2st']
    ens2pars_jk_meff3st[ens]=result['3st']

yu.makePDF('meff',figs)

label=f'ens2pars_jk_meffnst_selected'
yu.save_pkl_reg(label,[ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st])

e {'1st': 32, '2st': 11, '3st': 4}


In [5]:
# check error

def mom2num(mom):
    moms=yu.mom2moms(mom)
    return len(moms)

for ens in ['e']:
    c2pt=ens2c2pt[ens]
    moms=ens2moms_2pt[ens]
    msqs=[yu.mom2msq(mom) for mom in moms]
    msqs_unique=yu.removeDuplicates(msqs)
    msqs_do=[msq for msq in msqs_unique if msq<=25]
    
    for msq in msqs_do:
        inds=np.where(msqs==msq)[0]
        if len(inds)>1:
            ts=[5,15,25,35]
            c0=c2pt[:,ts,inds[0]]
            c1=c2pt[:,ts,inds[1]]
            
            m0=moms[inds[0]]; m1=moms[inds[1]]
            n0=mom2num(m0); n1=mom2num(m1)
            print(f'{m0} ({n0}); {m1} ({n1})')
            print(yu.jackme_un2str(c0))
            print(yu.jackme_un2str(c1))
            print(yu.jackme_un2str((c0+c1)/2))
            print(yu.jackme_un2str((c0*n0+c1*n1)/(n0+n1)))
            print()
    break

[0, 0, 3] (6); [1, 2, 2] (24)
[0.000000000020684(42), 0.0000000000008250(26), 0.00000000000004278(38), 0.00000000000000247(12)]
[0.000000000020679(42), 0.0000000000008253(27), 0.00000000000004273(35), 0.000000000000002384(90)]
[0.000000000020681(42), 0.0000000000008251(26), 0.00000000000004275(34), 0.000000000000002427(86)]
[0.000000000020680(42), 0.0000000000008252(26), 0.00000000000004274(34), 0.000000000000002402(83)]

[0, 1, 4] (24); [2, 2, 3] (24)
[0.000000000009151(20), 0.0000000000002372(11), 0.00000000000000815(17), 0.000000000000000288(54)]
[0.000000000009146(19), 0.0000000000002379(11), 0.00000000000000795(16), 0.000000000000000292(46)]
[0.000000000009148(19), 0.0000000000002376(10), 0.00000000000000805(15), 0.000000000000000290(39)]
[0.000000000009148(19), 0.0000000000002376(10), 0.00000000000000805(15), 0.000000000000000290(39)]

[1, 1, 4] (24); [0, 3, 3] (12)
[0.000000000008291(18), 0.00000000000020447(97), 0.00000000000000668(14), 0.000000000000000264(44)]
[0.000000000008

In [6]:
def run(ens):
    c2pt=ens2c2pt[ens]
    moms=ens2moms_2pt[ens]
    moms_sort=sorted(moms,key=yu.getSortkey_mom)
    
    # for mom in moms_sort:
    #     print(mom,yu.find_fitmax(c2pt[:,:,moms.index(mom)]))
    
    meff0=yu.jackmap(yu.c2pt2meff,c2pt[:,:,moms.index([0,0,0])])
    tmax=yu.find_fitmax(meff0)
    tmin=ens2selections[ens]['2st']
    ts=np.arange(tmin,tmax)
    y_jk=meff0[:,ts]
    def fitfunc(pars):
        return yu.func_meff_2st(ts,*pars)
    pars0=[0.3,0.2,0.8]
    pars_jk_0mom,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0)
    
    fitss=yu.load_pkl_reg('ens2pars_jk_meffnst_selected')
    if fitss is not None:
        [ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]=fitss
        print(ens,yu.jackme_un2str(pars_jk_0mom),yu.jackme_un2str(ens2pars_jk_meff2st[ens]))
        
    msqs=[yu.mom2msq(mom) for mom in moms]
    msqs_unique=yu.removeDuplicates(msqs)
    msqs_do=[msq for msq in msqs_unique if msq<=25]
    
    msq2pars_jk={}
    pars0=np.mean(pars_jk_0mom,axis=0)[1:]
    for msq in msqs_do:
        inds=np.where(np.array(msqs)==msq)[0]
        Ns=[mom2num(moms[ind]) for ind in inds]
        weights=np.array(Ns)/np.sum(Ns)
        
        t_c2pt=np.sum([c2pt[:,:,ind]*weights[i] for i,ind in enumerate(inds)],axis=0)

        parsExtra_jk=np.sqrt(pars_jk_0mom[:,:1]**2 + (2*np.pi/yu.ens2NL[ens])**2 * msq)
        def fitfunc(pars):
            dE1,rc1, E0=pars
            return yu.func_meff_2st(ts,E0,dE1,rc1)
        
        meff=yu.jackmap(yu.c2pt2meff,t_c2pt)
        tmax=yu.find_fitmax(meff)
        tmin=ens2selections[ens]['2st']
        ts=np.arange(tmin,tmax)
        y_jk=meff[:,ts]
        pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0,parsExtra_jk=parsExtra_jk)
        pars0=np.mean(pars_jk,axis=0)
        
        pars_jk=np.concatenate([parsExtra_jk,pars_jk],axis=1)
        msq2pars_jk[msq]=pars_jk
        
        # print(mom,yu.jackme_un2str(pars_jk))
    
    return msq2pars_jk

enss_do=enss
ens2msq2pars_jk=yu.load_pkl_reg('ens2msq2pars_jk')
if ens2msq2pars_jk is None:
    ens2msq2pars_jk={ens:{} for ens in enss_do}
    for ens in enss_do:
        ens2msq2pars_jk[ens]=run(ens)
    yu.save_pkl_reg('ens2msq2pars_jk',ens2msq2pars_jk)

In [7]:
def run(ens):
    c2pt=ens2c2pt[ens]
    moms=ens2moms_2pt[ens]
    moms_sort=sorted(moms,key=yu.getSortkey_mom)
    
    meff0=yu.jackmap(yu.c2pt2meff,c2pt[:,:,moms.index([0,0,0])])
    tmax=yu.find_fitmax(meff0)
    tmin=ens2selections[ens]['2st']
    ts=np.arange(tmin,tmax)
    y_jk=meff0[:,ts]
    def fitfunc(pars):
        return yu.func_meff_2st(ts,*pars)
    pars0=[0.3,0.2,0.8]
    pars_jk_0mom,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0)
    
    fitss=yu.load_pkl_reg('ens2pars_jk_meffnst_selected')
    if fitss is not None:
        [ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]=fitss
        print(ens,yu.jackme_un2str(pars_jk_0mom),yu.jackme_un2str(ens2pars_jk_meff2st[ens]))
    
    mom2pars_jk={}
    moms_do=moms_sort[:27] 
    pars0=np.mean(pars_jk_0mom,axis=0)[1:]
    for i in range(len(moms_do)):
        mom=moms_do[i]
        msq=yu.mom2msq(mom)
        parsExtra_jk=np.sqrt(pars_jk_0mom[:,:1]**2 + (2*np.pi/yu.ens2NL[ens])**2 * msq)
        def fitfunc(pars):
            dE1,rc1, E0=pars
            return yu.func_meff_2st(ts,E0,dE1,rc1)
        
        meff=yu.jackmap(yu.c2pt2meff,c2pt[:,:,moms.index(mom)])
        tmax=yu.find_fitmax(meff)
        tmin=ens2selections[ens]['2st']
        ts=np.arange(tmin,tmax)
        y_jk=meff[:,ts]
        pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0,parsExtra_jk=parsExtra_jk)
        pars0=np.mean(pars_jk,axis=0)
        
        pars_jk=np.concatenate([parsExtra_jk,pars_jk],axis=1)
        mom2pars_jk[tuple(mom)]=pars_jk
        
        # print(mom,yu.jackme_un2str(pars_jk))
    
    return mom2pars_jk

enss_do=enss[:3]
ens2mom2pars_jk=yu.load_pkl_reg('ens2mom2pars_jk')
if ens2mom2pars_jk is None:
    ens2mom2pars_jk={ens:{} for ens in enss_do}
    for ens in enss_do:
        ens2mom2pars_jk[ens]=run(ens)
    yu.save_pkl_reg('ens2mom2pars_jk',ens2mom2pars_jk)

In [8]:
fig,axs=yu.getFigAxs(2,len(enss_do),Lrow=4,Lcol=8,sharex=True,sharey='row')
yu.addColHeader(axs,[yu.ens2label[ens] for ens in enss_do])

axs[0,0].set_ylabel(r'$E_{0,1}(\vec{p})$ [GeV]')    
axs[1,0].set_ylabel(r'$rc1$')    
for iens,ens in enumerate(enss_do):
    yunit=yu.ens2aInv[ens]/1000
    mom2pars_jk=ens2mom2pars_jk[ens]
    moms=[tuple(mom) for mom in sorted(list(mom2pars_jk.keys()),key=yu.getSortkey_mom)]
    msqs=[yu.mom2msq(mom) for mom in moms]
    
    ax=axs[0,iens]
    
    t=np.transpose([mom2pars_jk[mom][:,0] for mom in moms])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    plt_x=yu.jitter_duplicate_x(plt_x,fraction=0.4)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='r',mfc='white')
    
    msqs_plt=np.arange(0,msqs[-1]+0.1,0.5)
    Es0=mom2pars_jk[(0,0,0)][:,0]
    Es_plt=np.transpose([np.sqrt(Es0**2+(2*np.pi/yu.ens2NL[ens])**2 * msq) for msq in msqs_plt])
    mean,err=yu.jackme(Es_plt*yunit)
    x=msqs_plt; ymin=mean-err; ymax=mean+err
    ax.plot(x,mean,color='r',linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color='r', alpha=0.2)
    
    t=np.transpose([mom2pars_jk[mom][:,0]+mom2pars_jk[mom][:,1] for mom in moms])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    plt_x=yu.jitter_duplicate_x(plt_x,fraction=0.4)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b',mfc='white')
    
    msqs_plt=np.arange(0,msqs[-1]+0.1,0.5)
    Es0=mom2pars_jk[(0,0,0)][:,0]+mom2pars_jk[(0,0,0)][:,1]
    Es_plt=np.transpose([np.sqrt(Es0**2+(2*np.pi/yu.ens2NL[ens])**2 * msq) for msq in msqs_plt])
    mean,err=yu.jackme(Es_plt*yunit)
    x=msqs_plt; ymin=mean-err; ymax=mean+err
    ax.plot(x,mean,color='b',linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color='b', alpha=0.2)
    
    ax=axs[1,iens]
    ax.set_xlabel('msq')
    
    t=np.transpose([mom2pars_jk[mom][:,2] for mom in moms])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean; plt_yerr=err
    plt_x=yu.jitter_duplicate_x(plt_x,fraction=0.4)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b',mfc='white')

for iens,ens in enumerate(enss_do):
    yunit=yu.ens2aInv[ens]/1000
    msq2pars_jk=ens2msq2pars_jk[ens]
    msqs=list(msq2pars_jk.keys()); msqs.sort()
    
    ax=axs[0,iens]
    
    t=np.transpose([msq2pars_jk[msq][:,0] for msq in msqs])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color='r')
    
    t=np.transpose([msq2pars_jk[msq][:,0]+msq2pars_jk[msq][:,1] for msq in msqs])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b')
    
    ax=axs[1,iens]
    
    t=np.transpose([msq2pars_jk[msq][:,2] for msq in msqs])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b')
    
yu.finalizePlot('GuidedFit_2st_allmom')

In [9]:
fig,axs=yu.getFigAxs(2,len(enss_do),Lrow=4,Lcol=8,sharex=True,sharey='row')
yu.addColHeader(axs,[yu.ens2label[ens] for ens in enss_do])

axs[0,0].set_ylabel(r'$E_{0,1}(\vec{p})$ [GeV]')    
axs[1,0].set_ylabel(r'$rc1$')    
for iens,ens in enumerate(enss_do):
    yunit=yu.ens2aInv[ens]/1000
    mom2pars_jk=ens2mom2pars_jk[ens]
    moms=[tuple(mom) for mom in sorted(list(mom2pars_jk.keys()),key=yu.getSortkey_mom)]
    msqs=[yu.mom2msq(mom) for mom in moms]
    
    ax=axs[0,iens]
    
    t=np.transpose([mom2pars_jk[mom][:,0] for mom in moms])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    plt_x=yu.jitter_duplicate_x(plt_x,fraction=0.4)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='r',mfc='white')
    
    msqs_plt=np.arange(0,msqs[-1]+0.1,0.5)
    Es0=mom2pars_jk[(0,0,0)][:,0]
    Es_plt=np.transpose([np.sqrt(Es0**2+(2*np.pi/yu.ens2NL[ens])**2 * msq) for msq in msqs_plt])
    mean,err=yu.jackme(Es_plt*yunit)
    x=msqs_plt; ymin=mean-err; ymax=mean+err
    ax.plot(x,mean,color='r',linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color='r', alpha=0.2)
    
    t=np.transpose([mom2pars_jk[mom][:,0]+mom2pars_jk[mom][:,1] for mom in moms])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    plt_x=yu.jitter_duplicate_x(plt_x,fraction=0.4)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b',mfc='white')
    
    msqs_plt=np.arange(0,msqs[-1]+0.1,0.5)
    Es0=mom2pars_jk[(0,0,0)][:,0]+mom2pars_jk[(0,0,0)][:,1]
    Es_plt=np.transpose([np.sqrt(Es0**2+(2*np.pi/yu.ens2NL[ens])**2 * msq) for msq in msqs_plt])
    mean,err=yu.jackme(Es_plt*yunit)
    x=msqs_plt; ymin=mean-err; ymax=mean+err
    ax.plot(x,mean,color='b',linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color='b', alpha=0.2)
    
    ax=axs[1,iens]
    ax.set_xlabel('msq')
    
    t=np.transpose([mom2pars_jk[mom][:,2] for mom in moms])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean; plt_yerr=err
    plt_x=yu.jitter_duplicate_x(plt_x,fraction=0.4)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b',mfc='white')

for iens,ens in enumerate(enss_do):
    yunit=yu.ens2aInv[ens]/1000
    msq2pars_jk=ens2msq2pars_jk[ens]
    msqs=list(msq2pars_jk.keys()); msqs.sort()
    
    ax=axs[0,iens]
    
    t=np.transpose([msq2pars_jk[msq][:,0] for msq in msqs])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color='r')
    
    t=np.transpose([msq2pars_jk[msq][:,0]+msq2pars_jk[msq][:,1] for msq in msqs])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b')
    
    ax=axs[1,iens]
    
    t=np.transpose([msq2pars_jk[msq][:,2] for msq in msqs])
    mean,err=yu.jackme(t)
    plt_x=msqs; plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color='b')
    
yu.finalizePlot('GuidedFit_2st_allmom')

In [10]:
overwrite=False
def run(ens,mom):
    moms_2pt=ens2moms_2pt[ens]
    meff=yu.jackmap(yu.c2pt2meff,ens2c2pt[ens][:,:,moms_2pt.index(mom)])
    
    tminss=ens2tminss[ens]
    selections=ens2selections[ens].copy()
    
    msq=mom[0]**2+mom[1]**2+mom[2]**2
    E0_ref=np.sqrt(yu.m_avgpn**2+ (2*np.pi/yu.ens2NL[ens]*yu.ens2aInv[ens])**2 * msq )/1000
    
    if msq>4:
        del selections['1st']
        del selections['3st']
        tminss=ens2tminss_large[ens]
    
    fitss_2pt=yu.doFits_meff_nst(meff,tminss,[0.4,0.5,2,0.8,1],downSampling=1,label=f'meff_{ens}_{yu.mom2str(mom)}',overwrite=overwrite,debugQ=True)
    fig,axd,result=yu.makePlot_2pt_SimoneStyle(meff,fitss_2pt,xunit=yu.ens2a[ens],yunit=yu.ens2aInv[ens]/1000,E0_ref=E0_ref,ylims='auto',\
        selection=selections,labelType='EN')
    fig.suptitle(rf'{yu.ens2label[ens]}; $\vec{{p}}=${mom}')
    yu.finalizePlot(closeQ=True)

    return fig,result


ens2nst2mom2pars={}

nsts=['1st','2st','3st']
for ens in enss[:3]:
    moms=sorted(ens2moms_2pt[ens],key=yu.getSortkey_mom)
    moms=[mom for mom in moms if yu.mom2msq(mom)<=20]
    msqs=np.array([yu.mom2msq(mom) for mom in moms])
    
    nst2mom2pars={nst:{} for nst in nsts}
    
    figs=[]
    for imom, mom in enumerate(moms):
        print(ens,f'{imom}/{len(moms)}',mom,end='               \r')
        # print(ens,f'{imom}/{len(moms)}',mom)
        fig,result=run(ens,mom)
        figs.append(fig)
        for nst in nsts:
            nst2mom2pars[nst][tuple(mom)]=result[nst]
    
    ENs=np.transpose([nst2mom2pars['2st'][tuple(mom)][:,0] for mom in moms])
    
    fig,axs=yu.getFigAxs(1,1)
    ax=axs[0,0]
    yunit=yu.ens2aInv[ens]/1000
    ax.set_xlabel('msq')
    ax.set_ylabel(r'$E_N(\vec{p})$')    
    
    mean,err=yu.jackme(ENs)
    plt_x=msqs; plt_y=mean*yunit; plt_yerr=err*yunit
    plt_x=yu.jitter_duplicate_x(plt_x)
    ax.errorbar(plt_x,plt_y,plt_yerr,color='r')
    
    msqs_plt=np.arange(0,msqs[-1]+0.1,0.5)
    mN=ENs[:,0]
    ENs_plt=np.transpose([np.sqrt(mN**2+(2*np.pi/yu.ens2NL[ens])**2 * msq) for msq in msqs_plt])
    mean,err=yu.jackme(ENs_plt*yunit)
    
    x=msqs_plt; ymin=mean-err; ymax=mean+err
    ax.plot(x,mean,color='r',linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color='r', alpha=0.2)
    yu.finalizePlot(closeQ=True)
    figs.insert(0,fig)
    
    yu.makePDF(f'Eeff_{ens}',figs)
    
    ens2nst2mom2pars[ens]=nst2mom2pars.copy()
    
yu.save_pkl_reg('ens2nst2mom2pars',ens2nst2mom2pars)

# 3pt

In [16]:
key2tf2ratio={}
for ens in enss:
    key2tf2ratio[(ens,'j+;conn')]={}
    key2tf2ratio[(ens,'j-;conn')]={}
    
    basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
    
    mN_jk=ens2pars_jk_meff2st[ens][:,0]
    factor_equal=1/(-3*mN_jk/4)
    
    path=f'{basepath}/conn_2pt{app}.h5'
    with h5py.File(path) as f:
        moms=yu.moms2list(f['moms'])
        imom=moms.index([0,0,0])
        
        tf2c2pt={}
        for tf in f['data'].keys():
            t=f[f'data/{tf}'][:]
            t=yu.jackknife(np.real(t[:,:,imom]))
            tf2c2pt[int(tf)]=t

    path=f'{basepath}/conn_0,0,0,0,0,0{app}.h5'
    with h5py.File(path) as f:
        moms=yu.moms2list(f['moms'])
        imom=moms.index([0,0,0,0,0,0])
        
        for jtf in f['data'].keys():
            j,tf=jtf.split('_'); tf=int(tf)
            t=f[f'data/{jtf}'][:]
            t=t[:,:,0,projs.index('P0'),inserts.index('tt')]
            c3pt=yu.jackknife(t)
            ratio=np.real(c3pt/tf2c2pt[tf][:,tf:tf+1]*factor_equal[:,None])
            key=(ens,j)
            key2tf2ratio[key][tf]=ratio
            
ens2tfs_conn={}
for ens in enss:
    tfs=list(key2tf2ratio[(ens,'j+;conn')].keys()); tfs.sort()
    ens2tfs_conn[ens]=tfs
    print(ens,tfs)
    
yu.save_pkl_reg('key2tf2ratio',key2tf2ratio)
yu.save_pkl_reg('ens2tfs_conn',ens2tfs_conn)

e [8, 11, 14, 17, 20, 23, 26]


In [15]:
key2bare={}

overwrite=False
symmetrizeQ=True
def createDic(key):
    ens,j=key
    gett=lambda t:round(t/yu.ens2a[ens])
    gett2=lambda t:round(t/yu.ens2a[ens]/2)*2

    tfmins_2st=ens2tfs_conn['e'][:-2]; tcmins_2st=range(2,ens2tfs_conn[ens][-1]//2-1)
    
    pars_jk_meff2st=ens2pars_jk_meff2st[ens]
    fittype='2st2step_SYMshare'
    
    tf2ratio=key2tf2ratio[(ens,j)]
    if symmetrizeQ:
        tf2ratio=yu.symmetrizeRatio(tf2ratio)
    fits_sum=yu.doFits_3pt('sum',tf2ratio,tfmins_2st,tcmins_2st,corrQ=False,label=f'{ens}_{j}_sum',overwrite=overwrite)
    fits_2st=yu.doFits_3pt(fittype,tf2ratio,tfmins_2st,tcmins_2st,pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,label=f'{ens}_{j}_2st_{symmetrizeQ}',overwrite=overwrite)
    # tfmin=gett2(8*yu.ens2a['b']); tcmin=gett(0.2)
    tfmin=gett2(0.7); tcmin=gett(0.2)
    # tfmin=gett2(1.0); tcmin=gett(0.2)
    fit_2st_MA=yu.doMA_3pt(fits_2st,fitlabels=(tfmin,tcmin))

    key2bare[(ens,j)]=fit_2st_MA[0][:,0]
    
    dic={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,None,None,fits_sum,fits_2st],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,fit_2st_MA],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,2,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,2,5,None,None],
        'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,gett(0.2),gett(0.2),None,None],
        'fit_2st_rainbow_midpoint:[fittype,pars_jk_meff2st]':[fittype,ens2pars_jk_meff2st[ens]],
        'xunit':yu.ens2a[ens],
    }
    return dic

js_plt=['j+;conn','j-;conn']; enss_plt=enss

for ij,j in enumerate(js_plt):
    print(f'{ij}/{len(js_plt)}',j,end='                 \r')
    keys=[(ens,j) for ens in enss_plt]

    list_dic=[createDic(key) for key in keys]

    fig,axs=yu.makePlot_3pt(list_dic,shows=['rainbow','midpoint','fit_2st','fit_sum'])
    yu.addRowHeader(axs,[yu.ens2label[ens] for ens in enss_plt])
    yu.finalizePlot(f'rainbow/{j}',mkdirQ=True)
    
yu.save_pkl_reg('key2bare_conn',key2bare)